In [15]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
from pathlib import Path


In [16]:
# =============================================================================
# ── 4. SILVER LAYER: Aggregation & Transformation (Commune level + Unpivot)
# =============================================================================
print("\n⚙️ Processing BRONZE -> SILVER...")

# Load Bronze
path_rhone_bronze = '../../../data/bronze/2017-pres-t2-commune-rhone-69-bronze.csv'

df_b = pd.read_csv(path_rhone_bronze, sep=";", dtype=str)

# --- A. Aggregate the Base Commune Metrics ---
# base_cols = ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune']
base_cols = ['Code du département', 'Libellé du département', 'Libellé de la commune']

metrics_cols = ['Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés']

# Convert metrics to float first to handle decimals like "257.0", then to integer for summing
for c in metrics_cols:
    df_b[c] = df_b[c].astype(float).astype(int)


⚙️ Processing BRONZE -> SILVER...


In [17]:
# Group by Commune and sum the metrics
df_commune = df_b.groupby(base_cols)[metrics_cols].sum().reset_index()

# Recalculate percentages correctly at the commune level
df_commune['% Abs/Ins'] = (df_commune['Abstentions'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Vot/Ins'] = (df_commune['Votants'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Blancs/Ins'] = (df_commune['Blancs'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Blancs/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Blancs'] / df_commune['Votants'] * 100), 0).round(2)
df_commune['% Nuls/Ins'] = (df_commune['Nuls'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Nuls/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Nuls'] / df_commune['Votants'] * 100), 0).round(2)
df_commune['% Exp/Ins'] = (df_commune['Exprimés'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Exp/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Exprimés'] / df_commune['Votants'] * 100), 0).round(2)



In [18]:
# --- B. Unpivot / Melt the Candidates Data ---
cand_dfs = []
for i in range(1, 3):
    # Extract columns for this specific candidate
    cand_subset = base_cols + [f'N°Panneau_{i}', f'Sexe_{i}', f'Nom_{i}', f'Prénom_{i}', f'Voix_{i}']
    temp = df_b[cand_subset].copy()
    
    # Standardize column names
    temp.rename(columns={
        f'N°Panneau_{i}': 'N°Panneau',
        f'Sexe_{i}': 'Sexe',
        f'Nom_{i}': 'Nom',
        f'Prénom_{i}': 'Prénom',
        f'Voix_{i}': 'Voix'
    }, inplace=True)
    cand_dfs.append(temp)



In [19]:
# Combine all candidates vertically
df_cands = pd.concat(cand_dfs, ignore_index=True)
df_cands['Voix'] = df_cands['Voix'].astype(float).astype(int)

# Group by Commune AND Candidate to sum their votes
cand_group_cols = base_cols + ['N°Panneau', 'Sexe', 'Nom', 'Prénom']
df_cands_grouped = df_cands.groupby(cand_group_cols)['Voix'].sum().reset_index()

# --- C. Merge & Finalize Silver Dataset ---
# Join candidate votes with commune total metrics
df_silver = pd.merge(df_cands_grouped, df_commune, on=base_cols, how='left')

# Recalculate Candidate Percentages
df_silver['% Voix/Ins'] = (df_silver['Voix'] / df_silver['Inscrits'] * 100).round(2)
df_silver['% Voix/Exp'] = np.where(df_silver['Exprimés'] > 0, (df_silver['Voix'] / df_silver['Exprimés'] * 100), 0).round(2)

In [20]:
# Reorder columns for readability
final_cols = base_cols + metrics_cols + [
    '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot',
    '% Nuls/Ins', '% Nuls/Vot', '% Exp/Ins', '% Exp/Vot',
    'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp'
]
df_silver = df_silver[final_cols]



In [22]:
def add_nuance_to_silver_data(df_silver, reference_file_path):
    import pandas as pd

    df_ref = pd.read_csv(reference_file_path, encoding="utf-8-sig").rename(columns={
        "Nom": "Prénom_ref",
        "Prénom": "Nom_ref"
    })

    for col in ["Nom", "Prénom"]:
        df_silver[col] = df_silver[col].astype(str).str.strip()

    for col in ["Nom_ref", "Prénom_ref"]:
        df_ref[col] = df_ref[col].astype(str).str.strip()

    df_ref = df_ref.drop_duplicates(subset=["Nom_ref", "Prénom_ref"])

    df_enriched = df_silver.merge(
        df_ref[["Nom_ref", "Prénom_ref", "nuance"]],
        left_on=["Nom", "Prénom"],
        right_on=["Nom_ref", "Prénom_ref"],
        how="left"
    ).drop(columns=["Nom_ref", "Prénom_ref"])

    return df_enriched

# Ajouter la nuance politique
reference_path = '../../../data/reference/nuance_politique_candidates_master.csv'
df_silver = add_nuance_to_silver_data(df_silver, reference_path)

# Vérifier que la colonne est bien présente
print(f"Colonnes finales: {list(df_silver.columns)}")
display(df_silver.head(5))

path_rhone_silver = '../../../data/silver/2017-pres-t2-commune-rhone-69-silver.csv'
df_silver.to_csv(path_rhone_silver, index=False, sep=";", encoding="utf-8")

print(f"✅ Updated SILVER dataset created: {path_rhone_silver}")
print(f"📊 Columns: {list(df_silver.columns)}")
print(f"📊 Total Rows: {len(df_silver)}")
display(df_silver.head(5))



Colonnes finales: ['Code du département', 'Libellé du département', 'Libellé de la commune', 'Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés', '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot', '% Nuls/Ins', '% Nuls/Vot', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'nuance_x', 'nuance_y']


,Code du département,Libellé du département,Libellé de la commune,Inscrits,Abstentions,Votants,Blancs,Nuls,Exprimés,% Abs/Ins,...,% Exp/Vot,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,nuance_x,nuance_y
0,69,Rhône,Affoux,257,39,218,20,7,191,15.18,...,87.61,1,M,MACRON,Emmanuel,93,36.19,48.69,REM,REM
1,69,Rhône,Affoux,257,39,218,20,7,191,15.18,...,87.61,2,F,LE PEN,Marine,98,38.13,51.31,RN,RN
2,69,Rhône,Aigueperse,224,51,173,13,11,149,22.77,...,86.13,1,M,MACRON,Emmanuel,84,37.50,56.38,REM,REM
3,69,Rhône,Aigueperse,224,51,173,13,11,149,22.77,...,86.13,2,F,LE PEN,Marine,65,29.02,43.62,RN,RN
4,69,Rhône,Albigny-sur-Saône,1665,402,1263,110,26,1127,24.14,...,89.23,1,M,MACRON,Emmanuel,775,46.55,68.77,REM,REM


✅ Updated SILVER dataset created: ../../../data/silver/2017-pres-t2-commune-rhone-69-silver.csv
📊 Columns: ['Code du département', 'Libellé du département', 'Libellé de la commune', 'Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés', '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot', '% Nuls/Ins', '% Nuls/Vot', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'nuance_x', 'nuance_y']
📊 Total Rows: 560


,Code du département,Libellé du département,Libellé de la commune,Inscrits,Abstentions,Votants,Blancs,Nuls,Exprimés,% Abs/Ins,...,% Exp/Vot,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,nuance_x,nuance_y
0,69,Rhône,Affoux,257,39,218,20,7,191,15.18,...,87.61,1,M,MACRON,Emmanuel,93,36.19,48.69,REM,REM
1,69,Rhône,Affoux,257,39,218,20,7,191,15.18,...,87.61,2,F,LE PEN,Marine,98,38.13,51.31,RN,RN
2,69,Rhône,Aigueperse,224,51,173,13,11,149,22.77,...,86.13,1,M,MACRON,Emmanuel,84,37.50,56.38,REM,REM
3,69,Rhône,Aigueperse,224,51,173,13,11,149,22.77,...,86.13,2,F,LE PEN,Marine,65,29.02,43.62,RN,RN
4,69,Rhône,Albigny-sur-Saône,1665,402,1263,110,26,1127,24.14,...,89.23,1,M,MACRON,Emmanuel,775,46.55,68.77,REM,REM
